# 05. 결함 분포 (1) — 기준 ④ 위험등급 미기재 원인 분해 (D63, 2026-11-23)

추천 상품 단위로 세 단계에 위험등급이 있는지 본다 (`src/d4_cause.py`).
- 원천: 상품 설명 속 운용사 등급 `body_grade` (D59)
- cot3: 후보 비교 단계에서 그 상품 뒤 구간에 숫자 등급 또는 공식 명칭
- answer: 최종 답변에서 같은 방식 (기준 ④ D19와 같은 함수)

조합 이름은 answer에 등급이 없는(기준 ④ 미충족) 추천 상품에만 붙인다.
- 원천 없음 → 데이터 생성 단계 / 원천 있음·cot3 있음 → 답변 정리 단계에서 빠짐 / 원천 있음·cot3 없음 → 추론 단계부터 빠짐

판정 없이 사실만 둔다. 원문 문장은 출력·저장하지 않고 cot_id와 상품명만 쓴다.

데이터: AI Hub(한국지능정보사회진흥원) 사업 결과

In [ ]:
import sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
from src.load_cot import load_cot, load_products
from src.judge import judge
from src.d4_cause import d4_cause, cot_level_d19
from src.report import update_scorecard
AS_OF = "2026-11-18"
cot = load_cot(); prod = load_products()
j, ctx = judge(cot, prod, as_of=AS_OF)
t = d4_cause(ctx, prod)
excl = set(pd.read_csv(ROOT / "outputs" / "exclusions.csv", encoding="utf-8-sig")["cot_id"])
t["제외 CoT"] = t["cot_id"].isin(excl)
print("추천 상품:", len(t), "/ CoT:", t["cot_id"].nunique(), "/ 분할:", t.groupby("분할")["cot_id"].nunique().to_dict())

## 1. 기준 ④ 판정과 대조 (answer 상태만으로 CoT 단위 재구성)

In [ ]:
MAP = {"전부 충족": "전부 기재", "일부": "일부 기재", "전무": "기재 없음"}
re4 = cot_level_d19(t).map(MAP)
cmp = j.set_index("cot_id")["기준④ 판정"].to_frame("기준④").join(re4.rename("재구성"))
cmp["재구성"] = cmp["재구성"].fillna("판정 불가(추천 상품 없음)")
print(pd.crosstab(cmp["기준④"], cmp["재구성"], margins=True))
assert (cmp["기준④"] == cmp["재구성"]).all(), "기준 ④ 판정과 다름"
print("일치: 전 CoT", len(cmp))

## 2. 단계별 상태 전체 조합 (추천 수 / CoT 수, 분할별)

CoT 수는 그 조합에 해당하는 추천 상품이 하나라도 있는 CoT 수다. 한 CoT가 여러 조합에 들어가므로 CoT 수의 합은 전체 CoT 수보다 크다.

In [ ]:
def by_split(df, keys):
    g = df.groupby(keys + ["분할"]).agg(추천=("상품명", "size"), CoT=("cot_id", "nunique")).unstack("분할", fill_value=0)
    g.columns = [f"{a}_{b}" for a, b in g.columns]
    tot = df.groupby(keys).agg(추천_전체=("상품명", "size"), CoT_전체=("cot_id", "nunique"))
    return g.join(tot).sort_values("추천_전체", ascending=False)
full = by_split(t, ["원천", "cot3", "answer"])
print(full.to_string())

## 3. answer 미기재 추천 상품의 원인 조합

In [ ]:
miss = t[t["answer"] != "있음"]
named = by_split(miss.assign(조합=miss["조합"].replace("", "이름 없음(원천 복수·매칭 안 됨)")), ["조합"])
print("answer 미기재 추천:", len(miss), "/ CoT:", miss["cot_id"].nunique()); print(named.to_string()); print()
print("[제외 CoT 뺀 뒤]"); keep = miss[~miss["제외 CoT"]]
print(by_split(keep.assign(조합=keep["조합"].replace("", "이름 없음(원천 복수·매칭 안 됨)")), ["조합"]).to_string()); print()
print("[원천 없음 추천의 등록 등급 유무] (전체 추천)")
print(pd.crosstab(t.loc[t["원천"] == "본문 등급 없음", "answer"], t.loc[t["원천"] == "본문 등급 없음", "등록 등급 있음"], margins=True))

## 4. answer 등급이 원천 등급과 다른 경우

숫자 등급(범위 표현 제외 첫 단일 등급)과 공식 명칭(구간에 하나만 나온 경우, 1 매우높은위험 … 6 매우낮은위험)을 따로 센다. 원천 등급이 있는 추천만 비교한다.

In [ ]:
for col in ["숫자 등급 비교", "명칭 등급 비교"]:
    print(f"[{col}]"); print(pd.crosstab(t[col], t["분할"], margins=True)); print()
diff = t[(t["숫자 등급 비교"] == "다름") | (t["명칭 등급 비교"] == "다름")]
print("둘 중 하나라도 다름: 추천", len(diff), "/ CoT", diff["cot_id"].nunique())
print("[숫자 등급 다름: 원천 → answer]")
nd = t[t["숫자 등급 비교"] == "다름"]
print(pd.crosstab(nd["원천 등급"].astype(int), nd["answer 숫자 등급"].astype(int), margins=True))
print("[예시 10건: cot_id·상품명만]")
print(diff[["cot_id", "분할", "상품명", "원천 등급", "answer 숫자 등급", "answer 명칭 등급"]].head(10).to_string(index=False))

## 5. 저장 (`outputs/d4_cause_by_product.csv`, `outputs/d4_cause_summary.csv`, scorecard)

In [ ]:
cols = ["cot_id", "분할", "상품명", "원천", "원천 등급", "등록 등급 있음", "cot3", "answer", "조합",
        "answer 숫자 등급", "숫자 등급 비교", "answer 명칭 등급", "명칭 등급 비교", "제외 CoT"]
t[cols].to_csv(ROOT / "outputs" / "d4_cause_by_product.csv", index=False, encoding="utf-8-sig")
full.to_csv(ROOT / "outputs" / "d4_cause_summary.csv", encoding="utf-8-sig")

def row(name, df, unit, how="추천"):
    f = (lambda d: len(d)) if how == "추천" else (lambda d: d["cot_id"].nunique())
    return {"지표": name, "Training": f(df[df["분할"] == "Training"]), "Validation": f(df[df["분할"] == "Validation"]),
            "전체": f(df), "단위": unit, "출처": "05_기준4_원인", "근거 결정": "D19·D59·D63", "기준일": AS_OF}
rows = [row("추천 상품", t, "추천"), row("answer 등급 미기재 추천", miss, "추천"), row("answer 등급 미기재 CoT", miss, "CoT", "CoT")]
for name in ["데이터 생성 단계", "답변 정리 단계에서 빠짐", "추론 단계부터 빠짐"]:
    rows += [row(f"미기재 원인: {name}", miss[miss["조합"] == name], "추천"),
             row(f"미기재 원인: {name}", miss[miss["조합"] == name], "CoT", "CoT")]
rows += [row("미기재 원인: 이름 없음(원천 복수·매칭 안 됨)", miss[miss["조합"] == ""], "추천"),
         row("answer 숫자 등급 ≠ 원천 등급", t[t["숫자 등급 비교"] == "다름"], "추천"),
         row("answer 명칭 등급 ≠ 원천 등급", t[t["명칭 등급 비교"] == "다름"], "추천")]
for lvl in ["전부 기재", "일부 기재", "기재 없음", "판정 불가(추천 상품 없음)"]:
    jj = j[j["기준④ 판정"] == lvl]
    rows.append({"지표": f"기준 ④ {lvl}", "Training": (jj["분할"] == "Training").sum(), "Validation": (jj["분할"] == "Validation").sum(),
                 "전체": len(jj), "단위": "CoT", "출처": "03_판정함수", "근거 결정": "D19", "기준일": AS_OF})
sc = update_scorecard(rows, "기준 ④ 위험등급 기재")
print(sc[sc["구분"] == "기준 ④ 위험등급 기재"][["지표", "Training", "Validation", "전체", "단위"]].to_string(index=False))

---
데이터: AI Hub(한국지능정보사회진흥원) 사업 결과